遍历目录下所有dataset4geodiff\out_openai_sematic_geodiff_txt，然后遍历第一级文件夹，文件夹名字就是apkname，

然后在文件夹里面，
找到以_threshold_mapping结尾的json文件，取出 path_mapping_results.all_kept_mappings.ours_id，这个值是clause的id，
再找到以_processed_converted结尾的json文件，取出app_info.baseline_version和app_info.target_version，
之后在这个文件夹dataset4geodiff/raw_geodiff_txt/{apkname}下找到以_country_versions结尾的csv文件，这里有baseline_version和REGIONS的对应关系，target_version和REGIONS的对应关系，

构建一个空的dataframe，横名是clause id，列名是regions，这样如果上面有出现clause id，就将target_version对应的国家的值填为1，其余为0即可


因为你现在这个 JSON 里没有“按地区分别判断”的字段，所以这 11 个地区列只能先同步填同一个值，也就是某个 clause 只要出现且为 1，这 11 列都填 1；否则填 0。这个判断和你上传的文件结构是一致的。


REGIONS = [
    "California", "Texas", "Germany", "Turkey", "Egypt",
    "Vietnam", "Nigeria", "India", "Saudi", "Bangladesh", "Pakistan"
]

In [ ]:
# B in : first_2,   first_third, first_4

In [11]:
# {f_position}\{s_position}
f_position = "AA1_first_batch"
s_position = "AA4_forth_100_batch" # AA1_first_328_batch, AA4_forth_100_batch, AA6_sisth_74_batch  ## AA1_first_328_batch, AA4_forth_100_batch, AA6_sisth_74_batch

# {parameter}
# parameter first_1,  first_2,   first_third, first_4, first_5, first_6
# second_1, second_2, second_3, second_4, second_5, second_6, second_7
# third_1,  third_2,  third_3,  third_4,  third_5,  third_6,  third_7
# fourth_1
parameter = "first_6"

In [12]:
from pathlib import Path
import json
import pandas as pd

# first_1,  first_2,   first_third, first_4, first_5, first_6
# second_1, second_2, second_3, second_4, second_5, second_6, second_7
# third_1,  third_2,  third_3,  third_4,  third_5,  third_6,  third_7
# fourth_1

ROOT_DIR = Path(r"dataset4geodiff/apk_versions_long.csv")
OUT_ROOT = Path(rf"dataset4geodiff/out_openai_sematic_geodiff_txt/{parameter}")
# RAW_ROOT = Path(r"dataset4geodiff/raw_geodiff_txt")
TEMPLATE_XLSX = Path(r"dataset4geodiff\can_detect_11regions_regulations_geodiff.xlsx")

# True: 某个 clause 出现就模板中的所有地区全置 1；False: 仅 target_version 对应地区置 1
USE_UNIFORM_REGION_FILL = False

In [13]:
def load_matrix_template_axes(template_xlsx_path: Path):
    tpl = pd.read_excel(template_xlsx_path, dtype=str)
    if tpl.empty:
        raise ValueError(f"Template xlsx is empty: {template_xlsx_path}")

    id_col = "Clause" if "Clause" in tpl.columns else tpl.columns[0]
    print(f"ID Column: {id_col}")
    clause_axis = [
        str(x).strip()
        for x in tpl[id_col].tolist()
        if str(x).strip() and str(x).strip().lower() != "nan"
    ]
    region_axis = [str(c).strip() for c in tpl.columns if c != id_col]

    if not clause_axis:
        raise ValueError(f"No clause rows found in template: {template_xlsx_path}")
    if not region_axis:
        raise ValueError(f"No region columns found in template: {template_xlsx_path}")
    print(f"Clause Axis: {clause_axis}\n", "len(clause_axis):", len(clause_axis))
    print(f"Region Axis: {region_axis}\n", "len(region_axis):", len(region_axis))
    return clause_axis, region_axis


TEMPLATE_CLAUSE_AXIS, TEMPLATE_REGION_AXIS = load_matrix_template_axes(TEMPLATE_XLSX)
REGIONS = TEMPLATE_REGION_AXIS
print(REGIONS)

ID Column: Clause
Clause Axis: ['P1', 'P2', 'P3', 'P4', 'P5', 'P6', 'P7', 'P8', 'P10', 'P11', 'P12', 'P13', 'P14', 'P15', 'P16', 'P17', 'P18', 'P19', 'P20', 'P21', 'CR1', 'CR2', 'CR3', 'CR4', 'CR6', 'C1', 'R1', 'R2', 'R4', 'R5', 'R6', 'R7', 'R8', 'R9', 'R10', 'R11', 'R13', 'R14', 'R15', 'R16', 'R17', 'R19', 'R20', 'E1', 'E24', 'E25', 'E26', 'E27', 'E28', 'O3']
 len(clause_axis): 50
Region Axis: ['California', 'Texas', 'Germany', 'Turkey', 'Egypt', 'Vietnam', 'Nigeria', 'India', 'Saudi', 'Bangladesh', 'Pakistan']
 len(region_axis): 11
['California', 'Texas', 'Germany', 'Turkey', 'Egypt', 'Vietnam', 'Nigeria', 'India', 'Saudi', 'Bangladesh', 'Pakistan']


In [11]:
# # find all versions of each apk in the dataset
# apk_version_df = pd.read_csv("dataset4geodiff/apk_versions_long.csv")
# apk_version_df.head(), apk_version_df.shape

# example: find all versions of HinKhoj.Dictionary
# for version in apk_version_df[apk_version_df['apkname'] == 'HinKhoj.Dictionary']["version"].values:
#     print(str(version).strip())

(                    apkname  version
 0        HinKhoj.Dictionary      310
 1        HinKhoj.Dictionary      320
 2  ae.brandsforless.android      412
 3  ae.brandsforless.android      413
 4       air.bg.lan.Monopoli  7000009,
 (2193, 2))

In [14]:
def process_version(apk_version_df, apkname, baseline_version):
    apk_rows = apk_version_df[apk_version_df['apkname'] == apkname]
    if apk_rows.empty:
        print(f"[WARN] {apkname} not found in version mapping file")
        return None
    versions = apk_rows["version"].values
    for v in versions:
        if str(v).strip()==baseline_version:
            continue
        target_version = str(v).strip()
    if not target_version:
        print(f"[WARN] {apkname} has empty version in mapping file")
        return None

    return target_version

def find_one_file(folder: Path, pattern: str):
    matches = sorted(folder.glob(pattern))
    return matches[0] if matches else None


def _extract_mapping_items(obj):
    if isinstance(obj, dict):
        all_kept = obj.get("all_kept_mappings", [])
        if isinstance(all_kept, list):
            return all_kept
        return []
    if isinstance(obj, list):
        items = []
        for x in obj:
            if isinstance(x, dict):
                if "all_kept_mappings" in x:
                    v = x.get("all_kept_mappings", [])
                    if isinstance(v, list):
                        items.extend(v)
                else:
                    items.append(x)
        return items
    return []




def load_clause_ids(threshold_mapping_path: Path):
    with open(threshold_mapping_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    items = []
    if isinstance(data, dict):
        pmr = data.get("path_mapping_results", {})
        items = _extract_mapping_items(pmr)
    elif isinstance(data, list):
        for entry in data:
            if not isinstance(entry, dict):
                continue
            if "path_mapping_results" in entry:
                items.extend(_extract_mapping_items(entry.get("path_mapping_results")))
            else:
                items.extend(_extract_mapping_items(entry))

    clause_ids = []
    for item in items:
        if not isinstance(item, dict):
            continue
        cid = str(item.get("ours_id", "")).strip()
        if cid:
            clause_ids.append(cid)
    return sorted(set(clause_ids))


def load_versions_1(processed_converted_path: Path):
    with open(processed_converted_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    app_info = data.get("app_info", {})
    baseline_version = str(app_info.get("baseline_version", "")).strip()
    target_version = str(app_info.get("target_version", "")).strip()
    return baseline_version, target_version

def find_target_version(all_info_df: pd.DataFrame, app_id: str, baseline: str) -> str:
    # 从所有 apk 版本信息里，找到一个与 baseline 不同的版本
    apk = all_info_df[all_info_df["apkname"] == app_id]
    if apk.empty:
        return ""
    vals = apk["version"].astype(str).str.strip().unique().tolist()
    uniq = []
    for v in vals:
        if v not in uniq:
            if v != baseline:
                uniq.append(v)
    print(app_id,uniq)
    return uniq[0] if uniq else ""

def load_versions(ROOT_DIR: Path, country_csv_path: Path, apkname: str):
    all_apks_info = pd.read_csv(ROOT_DIR, dtype=str)
    # 1) 读取 country_versions.csv
    try:
        df = pd.read_csv(country_csv_path, dtype=str).fillna("")
    except Exception as e:
        print(f"[跳过] {country_csv_path}: 读取 csv 失败 -> {e}")
        return "", ""

    if df.empty:
        print(f"[跳过] {country_csv_path}: csv 为空")
        return "", ""

    row = df.iloc[0]
    baseline_version = str(row.get("usa_version_code", "")).strip()

    target_version = find_target_version(all_apks_info, apkname, baseline_version)

    return baseline_version, target_version

REGION_COLUMN_ALIASES = {
    "California": ["usa_version_code"],
    "Texas": ["usa_version_code"],
    "Germany": ["germany_version_code"],
    "Turkey": ["turkey_version_code"],
    "Egypt": ["egypt_version_code"],
    "Vietnam": ["vietnam_version_code"],
    "Nigeria": ["nigeria_version_code"],
    "India": ["india_version_code"],
    "Saudi": ["saudi_version_code"],
    "Bangladesh": ["bangladesh_version_code"],
    "Pakistan": ["pakistan_version_code"],
}


def get_region_candidate_columns(region: str):
    aliases = REGION_COLUMN_ALIASES.get(region, [])
    default_col = f"{region.lower()}_version_code"
    # print(f"Region '{region}' candidate columns: {aliases + [default_col]}")
    return list(dict.fromkeys(aliases + [default_col]))


def regions_for_version(country_df: pd.DataFrame, version: str):
    target_regions = []
    v = str(version).strip()
    if not v:
        return target_regions

    for region in REGIONS:
        candidate_cols = get_region_candidate_columns(region)
        matched = False
        for col in candidate_cols:
            if col not in country_df.columns:
                continue
            series = country_df[col].astype(str).str.strip()
            if (series == v).any():
                matched = True
                break
        if matched:
            target_regions.append(region)
    return target_regions


def build_matrix(detected_clause_ids, active_regions):
    matrix = pd.DataFrame(0, index=TEMPLATE_CLAUSE_AXIS, columns=TEMPLATE_REGION_AXIS, dtype=int)
    matrix.index.name = "Clause" # "clause_id"

    clause_set = set(detected_clause_ids)
    region_set = set(active_regions)

    for cid in TEMPLATE_CLAUSE_AXIS:
        if cid not in clause_set:
            continue
        for region in TEMPLATE_REGION_AXIS:
            if region in region_set:
                matrix.at[cid, region] = 1
    return matrix


def process_one_apk_folder(apk_folder: Path):
    apkname = apk_folder.name

    

    threshold_mapping_path = find_one_file(apk_folder, "*process.json")
    if threshold_mapping_path is None:
        threshold_mapping_path = find_one_file(apk_folder, "*process*.json")

    # processed_converted_path = find_one_file(apk_folder, "*_processed_converted.json")

    raw_apk_folder = OUT_ROOT / apkname
    country_csv_path = find_one_file(raw_apk_folder, "*_country_versions.csv")

    print("threshold_mapping_path", threshold_mapping_path)
    if threshold_mapping_path is None:
        print(f"[SKIP] {apkname}: threshold mapping json not found")
        return
    # if processed_converted_path is None:
    #     print(f"[SKIP] {apkname}: processed_converted json not found")
    #     return
    if country_csv_path is None:
        print(f"[SKIP] {apkname}: country_versions csv not found")
        return

    clause_ids = load_clause_ids(threshold_mapping_path)
    print(f"[INFO] {apkname}: detected clause ids: {clause_ids}")
    baseline_version, target_version = load_versions(ROOT_DIR, country_csv_path, apkname)
    print(f"[INFO] {apkname}: US version is baseline_version={baseline_version}, target_version={target_version}")

    country_df = pd.read_csv(country_csv_path, dtype=str)
    country_df.drop(columns=['package_name'], inplace=True)
    # print(f"{country_df.head()}")



    target_regions = regions_for_version(country_df, target_version)
    # print(apkname, target_regions)

    if USE_UNIFORM_REGION_FILL:
        active_regions = TEMPLATE_REGION_AXIS if clause_ids else []
    else:
        active_regions = target_regions

    print("active_regions", active_regions)

    matrix_df = build_matrix(clause_ids, active_regions)
    # 删除行索引为 "P9" 的数据
    # matrix_df = matrix_df.drop("P9")
    print(matrix_df.shape)


    out_path = apk_folder / f"{apkname}_clause_region_matrix.csv"

    matrix_df.to_csv(out_path, encoding="utf-8-sig")

    print(
        f"[OK] {apkname}: clauses_detected={len(clause_ids)}, "
        f"template_rows={len(TEMPLATE_CLAUSE_AXIS)}, template_cols={len(TEMPLATE_REGION_AXIS)}, "
        f"target_version={target_version}, target_regions={target_regions}, saved={out_path}"
    )

In [15]:
# # 批量遍历第一级 APK 文件夹
for apk_folder in sorted([p for p in OUT_ROOT.iterdir() if p.is_dir()]):
    apkname = apk_folder.name
    # skip folders that are not APK package names
    if "." not in apkname:
        print(f"[SKIP] Not an APK folder: {apkname}")
        continue

    print(f"Processing APK folder: {apk_folder}")
    process_one_apk_folder(apk_folder)
# com.benoitletondor.pixelminimalwatchface
# process_one_apk_folder(Path(r"dataset4geodiff\out_openai_sematic_geodiff_txt\air.com.bigwigmedia.hotdogbush")) # com.bandagames.mpuzzle.gp ae.brandsforless.android  air.bg.lan.Monopoli  air.com.bigwigmedia.hotdogbush

## 不用

<!-- 遍历目录下所有dataset4geodiff\out_openai_sematic_geodiff_txt，然后遍历第一级文件夹，文件夹名字就是apkname，

然后在文件夹里面，
找到以_clause_region_matrix结尾的csv文件，加载为dataframe1

在这个路径下D:\AA_project\AA_geo_app_regulation\Evaluation\dataset以AA开头的一级文件夹里，打开名为matrix_output的文件夹，找到文档名为该apkname的文档，加载为dataframe2

合并dataframe1和dataframe2，得到完整的final_dataframe，保存为csv -->

In [ ]:
# import os
# from pathlib import Path
# from typing import Optional
# import pandas as pd

In [ ]:
# # ========= 路径配置 =========
# base_dir1 = Path(r"dataset4geodiff\out_openai_sematic_geodiff_txt")
# base_dir2 = Path(r"D:\AA_project\AA_geo_app_regulation\Evaluation\dataset")
# output_dir = Path(rf"{base_dir2}\merged_final_dataframe")

# output_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
# # ========= 工具函数 =========
# def find_clause_region_matrix_csv(apk_dir: Path):
#     """
#     在 apk_dir 中寻找以 _clause_region_matrix 结尾的 csv 文件
#     """
#     candidates = list(apk_dir.glob("*_clause_region_matrix.csv"))
#     if len(candidates) == 0:
#         return None
#     if len(candidates) > 1:
#         print(f"[WARN] {apk_dir.name} 下找到多个 *_clause_region_matrix.csv，默认使用第一个: {candidates[0].name}")
#     return candidates[0]



# def find_matrix_output_file(apkname: str, dataset_root: Path) -> Optional[Path]:
#     """
#     在 dataset_root 下所有以 AA 开头的一级文件夹中，
#     遍历其下一级子文件夹，
#     再进入子文件夹中的 matrix_output 文件夹，
#     寻找文件名为 apkname 的文件。
    
#     支持: csv / xlsx / xls
#     """
#     exts = [".csv", ".xlsx", ".xls"]

#     for aa_dir in dataset_root.iterdir():
#         if not aa_dir.is_dir() or not aa_dir.name.startswith("AA"):
#             continue

#         # 遍历 AA 文件夹下的一级子文件夹
#         for sub_dir in aa_dir.iterdir():
#             if not sub_dir.is_dir():
#                 continue

#             matrix_output_dir = sub_dir / "matrix_output"
#             if not matrix_output_dir.is_dir():
#                 continue

#             for ext in exts:
#                 candidate = matrix_output_dir / f"{apkname}{ext}"
#                 if candidate.is_file():
#                     return candidate

#     return None

# def find_matrix_output_file_1(apkname: str, dataset_root: Path):
#     """
#     在 dataset_root 下所有以 AA 开头的一级文件夹中，
#     遍历这些 AA 文件夹下面的一级子文件夹，
#     再进入 matrix_output 文件夹，寻找文件名为 apkname 的文件
#     支持 csv/xlsx/xls
#     """
#     exts = [".csv", ".xlsx", ".xls"]

#     aa_dirs = [p for p in dataset_root.iterdir() if p.is_dir() and p.name.startswith("AA")]
#     for aa_dir in aa_dirs:
#         matrix_output_dir = aa_dir / "matrix_output"
#         if not matrix_output_dir.exists() or not matrix_output_dir.is_dir():
#             continue

#         for ext in exts:
#             candidate = matrix_output_dir / f"{apkname}{ext}"
#             if candidate.exists():
#                 return candidate

#         # 如果不是严格同名+扩展名，也可以尝试模糊匹配
#         fuzzy_candidates = []
#         for f in matrix_output_dir.iterdir():
#             if f.is_file() and f.stem == apkname and f.suffix.lower() in exts:
#                 fuzzy_candidates.append(f)

#         if len(fuzzy_candidates) > 0:
#             if len(fuzzy_candidates) > 1:
#                 print(f"[WARN] 在 {matrix_output_dir} 中为 {apkname} 找到多个候选文件，默认使用第一个: {fuzzy_candidates[0].name}")
#             return fuzzy_candidates[0]

#     return None

# def load_dataframe(file_path: Path):
#     """
#     根据扩展名加载 dataframe
#     """
#     suffix = file_path.suffix.lower()
#     if suffix == ".csv":
#         return pd.read_csv(file_path, index_col=0)
#     elif suffix in [".xlsx", ".xls"]:
#         return pd.read_excel(file_path)
#     else:
#         raise ValueError(f"不支持的文件类型: {file_path}")


# def merge_dataframes(df1: pd.DataFrame, df2: pd.DataFrame) -> pd.DataFrame:
#     # 复制，避免修改原表
#     df1 = df1.copy()
#     df2 = df2.copy()

#     # 清洗行名和列名
#     df1.index = df1.index.map(lambda x: str(x).strip())
#     df2.index = df2.index.map(lambda x: str(x).strip())
#     df1.columns = [str(c).strip() for c in df1.columns]
#     df2.columns = [str(c).strip() for c in df2.columns]

#     # 转成数值，非法值会报错
#     df1 = df1.apply(pd.to_numeric, errors="raise")
#     df2 = df2.apply(pd.to_numeric, errors="raise")

#     # 按 index 和 columns 自动对齐，缺失补 0
#     all_index = df1.index.union(df2.index)
#     all_columns = df1.columns.union(df2.columns)

#     df1 = df1.reindex(index=all_index, columns=all_columns, fill_value=0)
#     df2 = df2.reindex(index=all_index, columns=all_columns, fill_value=0)

#     # 转 int 后做并集
#     final_df = df1.fillna(0).astype(int) | df2.fillna(0).astype(int)

#     # 如果你想保持原始顺序而不是字母排序，可以再单独处理
#     return final_df

# def merge_dataframes_1(df1: pd.DataFrame, df2: pd.DataFrame):
#     # 检查行名和列名是否一致
#     if list(df1.index) != list(df2.index):
#         raise ValueError("df1 和 df2 的行名（Clause）不一致")
#     if list(df1.columns) != list(df2.columns):
#         raise ValueError("df1 和 df2 的列名（regions）不一致")

#     # 统一转成数值型 0/1
#     df1_bin = df1.fillna(0).astype(int)
#     df2_bin = df2.fillna(0).astype(int)

#     # 做并集（逐元素 OR）
#     final_df = df1_bin | df2_bin
#     return final_df

# # ========= 主流程 =========
# def main():
#     if not base_dir1.exists():
#         raise FileNotFoundError(f"找不到目录: {base_dir1}")
#     if not base_dir2.exists():
#         raise FileNotFoundError(f"找不到目录: {base_dir2}")

#     apk_dirs = [p for p in base_dir1.iterdir() if p.is_dir()]
#     print(f"共找到 {len(apk_dirs)} 个 apk 文件夹")

#     success_count = 0
#     fail_count = 0
#     failed_apks = []

#     for apk_dir in apk_dirs:
#         apkname = apk_dir.name
#         print(f"\n[INFO] 正在处理: {apkname}")

#         try:
#             # 1. 找 dataframe1
#             df1_file = find_clause_region_matrix_csv(apk_dir)
#             if df1_file is None:
#                 print(f"[WARN] 未找到 *_clause_region_matrix.csv: {apkname}")
#                 fail_count += 1
#                 failed_apks.append((apkname, "missing dataframe1"))
#                 continue

#             # 2. 找 dataframe2
#             df2_file = find_matrix_output_file(apkname, base_dir2)
#             if df2_file is None:
#                 print(f"[WARN] 未找到 matrix_output 中名为 {apkname} 的文件")
#                 fail_count += 1
#                 failed_apks.append((apkname, "missing dataframe2"))
#                 continue

#             print(f"[INFO] dataframe1: {df1_file}")
#             print(f"[INFO] dataframe2: {df2_file}")

#             # 3. 加载
#             df1 = load_dataframe(df1_file)
#             print(f"[INFO] dataframe1 shape: {df1.shape}, columns: {df1.columns.tolist()}")
#             df2 = load_dataframe(df2_file)
#             print(f"[INFO] dataframe2 shape: {df2.shape}, columns: {df2.columns.tolist()}")

#             # 4. 合并
#             final_df = merge_dataframes(df1, df2)
#             # final_df = df1.astype(int) | df2.astype(int)

#             # 5. 保存
#             save_path = output_dir / f"{apkname}_final_dataframe.csv"
#             # final_df.to_csv(save_path, index=False, encoding="utf-8-sig")
#             final_df.to_csv(save_path, encoding="utf-8-sig", index=True, index_label="Clause")

#             print(f"[OK] 已保存: {save_path}")
#             success_count += 1

#         except Exception as e:
#             print(f"[ERROR] 处理 {apkname} 失败: {e}")
#             fail_count += 1
#             failed_apks.append((apkname, str(e)))

#     print("\n========== 处理完成 ==========")
#     print(f"成功: {success_count}")
#     print(f"失败: {fail_count}")

#     if failed_apks:
#         fail_log = output_dir / "failed_apks.csv"
#         pd.DataFrame(failed_apks, columns=["apkname", "reason"]).to_csv(
#             fail_log, index=False, encoding="utf-8-sig"
#         )
#         print(f"失败记录已保存到: {fail_log}")

# if __name__ == "__main__":
#     main()

共找到 2 个 apk 文件夹

[INFO] 正在处理: com.bandagames.mpuzzle.gp
[INFO] dataframe1: dataset4geodiff\out_openai_sematic_geodiff_txt\com.bandagames.mpuzzle.gp\com.bandagames.mpuzzle.gp_clause_region_matrix.csv
[INFO] dataframe2: D:\AA_project\AA_geo_app_regulation\Evaluation\dataset\AA1_first_batch\AA2_second_100_batch\matrix_output\com.bandagames.mpuzzle.gp.csv
[INFO] dataframe1 shape: (51, 11), columns: ['California', 'Texas', 'Germany', 'Turkey', 'Egypt', 'Vietnam', 'Nigeria', 'India', 'Saudi', 'Bangladesh', 'Pakistan']
[INFO] dataframe2 shape: (51, 11), columns: ['California', 'Texas', 'Germany', 'Turkey', 'Egypt', 'Vietnam', 'Nigeria', 'India', 'Saudi', 'Bangladesh', 'Pakistan']
[OK] 已保存: D:\AA_project\AA_geo_app_regulation\Evaluation\dataset\merged_final_dataframe\com.bandagames.mpuzzle.gp_final_dataframe.csv

[INFO] 正在处理: com.benoitletondor.pixelminimalwatchface
[INFO] dataframe1: dataset4geodiff\out_openai_sematic_geodiff_txt\com.benoitletondor.pixelminimalwatchface\com.benoitletondor.pix